# Why MLA exists

This notebook compares logical KV-cache storage for MHA, GQA, MQA, and a toy MLA configuration.

In [1]:
import sys
from pathlib import Path

project_root = next(
    (candidate for candidate in (Path.cwd(), *Path.cwd().parents)
     if (candidate / "src" / "model").is_dir()),
    None,
)
if project_root is None:
    raise RuntimeError("Could not find the repository root")
sys.path.insert(0, str(project_root))

## Compare cache sizes

The toy MLA cache stores one shared latent and one positional key per token and layer.

In [2]:
from src.inference.kv_cache_memory import estimate_cache_mib

settings = {
    "batch_size": 1,
    "seq_len": 4096,
    "n_layers": 6,
    "query_heads": 8,
    "head_dim": 32,
    "gqa_kv_heads": 2,
    "kv_latent_dim": 64,
    "rope_dim": 16,
    "bytes_per_element": 2,
}

for attention_type in ("mha", "gqa", "mqa", "mla"):
    memory = estimate_cache_mib(attention_type, **settings)
    print(f"{attention_type}: {memory:.2f} MiB")

mha: 24.00 MiB
gqa: 6.00 MiB
mqa: 3.00 MiB
mla: 3.75 MiB


## Scaling behavior

Cache storage grows linearly with batch size, sequence length, and layer count. Doubling any one of them doubles the logical cache size.

In [3]:
base = estimate_cache_mib("mha", **settings)
longer = estimate_cache_mib("mha", **{**settings, "seq_len": 8192})
larger_batch = estimate_cache_mib("mha", **{**settings, "batch_size": 2})

print("base:", base)
print("longer context:", longer)
print("larger batch:", larger_batch)
assert longer == 2 * base
assert larger_batch == 2 * base

base: 24.0
longer context: 48.0
larger batch: 48.0
